# 🕵️ Baseball Lab 2: Stat Detective
Today you're a detective. Your evidence: **every MLB batter's every season, from 1871 to today.**

First a short Python warm-up, so you know exactly what every symbol does. Then the cases.
It takes about **45 minutes**, and you can stop after any part.

**How it works:**
- Click a gray box and press **Shift + Enter** to run it.
- `___` (three underscores) means **you fill it in**. The box won't work until you do.
- 🔎 **Google it** gives you words to search if you get stuck. Googling is not cheating:
  programmers google things all day long.
- A red error is normal. Read the **last line** of it: that's where Python says what went wrong.

---
## Part 1. Python warm-up 🔥

### `print` shows things on the screen

In [ ]:
print("Home run!")

Three things are happening here:
- `print` is a **command**. It tells Python: show something on the screen.
- The **round brackets** `( )` hold what you give to the command, like handing it the ball.
  A command always needs its brackets, even when they're empty.
- The **quotes** `" "` mean: this is text, show it exactly as it is.

What if we forget the quotes? Run this box. It will show an error, **on purpose**:

In [ ]:
print(Homerun)

Look at the last line of the error: `NameError: name 'Homerun' is not defined`.
Without quotes, Python thinks `Homerun` is the **name** of something it should already know,
and it doesn't know anything called that. So:
**text you want to show → in quotes.**

### Variables: a box with a name on it
A **variable** is like a locker with a name tag. You put something inside, and later you ask for it by its name.
The `=` sign means "put this into the box".

In [ ]:
player = "Aaron Judge"

print(player)
print("player")

Look at the two lines you got:
- `print(player)`, **no quotes**: Python opens the box called `player` and shows what's inside → **Aaron Judge**.
- `print("player")`, **with quotes**: Python just shows the word itself → **player**.

That's why we don't put variable names in quotes: then Python would show the name, not what's in the box.

You can show text and a variable together. Put a **comma** between them:

In [ ]:
player = "Aaron Judge"
number = 99

print("My favorite player is", player, "and he wears number", number)

Notice: `99` has **no quotes**. Numbers don't need them, and Python can do math with them.
Text in quotes is just letters, even if those letters look like numbers. Try it:

In [ ]:
print(100 + 50)
print("100" + "50")

The first line does math: **150**. The second just glues two pieces of text together: **10050**. 😄

### ⚾ Your turn
Change the name and the number to **your** favorite player, then run it.
Remember: the name goes in quotes, the number doesn't.

In [ ]:
player = "Aaron Judge"   # change to your favorite player
number = 99              # change to his number

print("My favorite player is", player, "and he wears number", number)

---
## Part 2. Open the evidence box 📦
The evidence comes from two places: the **Lahman Baseball Database** (a history book of baseball, 1871 to last year)
and **MLB's own live stats** for the newest season. Gluing them together takes a lot of code, so we hid it.
Click the ▶ next to the title and wait about 20 seconds.

It starts with a line you'll see a lot: `import pandas as pd`. That brings in a helper called **pandas**,
which lets Python work with tables, like a spreadsheet. `as pd` gives it a short nickname,
so we can type `pd` instead of `pandas`. (Double-click the title to see all the code.)

In [ ]:
#@title ⚾ Load the MLB data, 1871 to today (click ▶ and wait about 20 seconds)
import datetime
import unicodedata

import pandas as pd
import requests

LAHMAN = "https://raw.githubusercontent.com/daviddalpiaz/pylahman/main/data-raw/"
MLB = "https://statsapi.mlb.com/api/v1/"
STATS = ["G", "AB", "H", "HR", "RBI", "SB", "BB", "SO"]


def plain(text):
    # "Acuña" -> "Acuna": kids type names on an English keyboard
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()


def short(text):
    # "Acuña Jr." -> "acu": the start of a last name, to match players across the two sources
    return plain(text).lower()[:3]


# History: the Lahman Baseball Database (SABR, CC BY-SA 3.0). One row per player, season and team.
batting = pd.read_csv(LAHMAN + "Batting.csv")
people = pd.read_csv(LAHMAN + "People.csv")
teams = pd.read_csv(LAHMAN + "Teams.csv")[["yearID", "teamID", "name"]].rename(columns={"name": "team"})
batting = batting.merge(teams, on=["yearID", "teamID"], how="left")

# A player traded during a season has a row for each team: add them up and keep his last team.
per_season = batting.groupby(["playerID", "yearID"], sort=False)
history = per_season[STATS].sum(min_count=1).join(per_season["team"].last()).reset_index()
people["name"] = people["nameFirst"] + " " + people["nameLast"]
history = history.merge(people[["playerID", "name"]], on="playerID").rename(columns={"yearID": "season"})

# The seasons the history book doesn't have yet come live from MLB (statsapi.mlb.com).
lahman_ids = {
    (int(p.birthYear), int(p.birthMonth), int(p.birthDay), short(p.nameLast)): p.playerID
    for p in people.dropna(subset=["birthYear", "birthMonth", "birthDay"]).itertuples()
}
mlb_names, live = {}, []
try:
    for season in range(int(history["season"].max()) + 1, datetime.date.today().year + 1):
        url = MLB + f"stats?stats=season&group=hitting&season={season}&sportId=1&playerPool=ALL&limit=5000"
        found = requests.get(url, timeout=60).json().get("stats", [])
        hitters = found[0]["splits"] if found else []
        if not hitters:
            continue
        roster = requests.get(MLB + f"sports/1/players?season={season}", timeout=60).json()["people"]
        born = {p["id"]: p.get("birthDate", "") for p in roster}
        for h in hitters:
            p, s = h["player"], h["stat"]
            y, m, d = (born.get(p["id"]) or "0-0-0").split("-")
            player_id = lahman_ids.get((int(y), int(m), int(d), short(p["lastName"])), f"mlb{p['id']}")
            mlb_names[player_id] = p["fullName"]
            live.append({
                "playerID": player_id, "season": season, "team": h.get("team", {}).get("name"), "name": p["fullName"],
                "G": s["gamesPlayed"], "AB": s["atBats"], "H": s["hits"], "HR": s["homeRuns"], "RBI": s["rbi"],
                "SB": s["stolenBases"], "BB": s["baseOnBalls"], "SO": s["strikeOuts"],
            })
except (requests.RequestException, ValueError, KeyError) as problem:
    live = []
    print("MLB's live stats didn't answer, so the newest season is missing today. Everything else works.", problem)

stats = pd.concat([history, pd.DataFrame(live)], ignore_index=True)
# Players active today get the name MLB uses now on their whole career ("Vladimir Guerrero Jr."), without accents.
stats["name"] = stats["playerID"].map(mlb_names).fillna(stats["name"]).map(plain)
# Whole numbers (114, not 114.0); a stat nobody counted back then stays empty: <NA>.
stats[STATS] = stats[STATS].astype("Int64")
stats = stats.sort_values(["playerID", "season"], ignore_index=True)
stats = stats[["name", "season", "team"] + STATS + ["playerID"]]
print("Ready!", len(stats), "rows, seasons", stats["season"].min(), "to", stats["season"].max())

The hidden box made one big table called `stats`. This line shows its first 5 rows, with only the 4 columns we care about:

In [ ]:
stats[["name", "season", "team", "HR"]].head()

That's the top of the table. Two words you'll need:
- A **row** is one line across: **one player in one season**.
- A **column** is one kind of number going down, like `HR`.

**Square brackets** pick columns from the table:
- `stats["HR"]`: one column, its name in quotes inside `[ ]`.
- `stats[["season", "HR"]]`: **double** brackets hold a list of several columns to show.

The columns we'll use today:

| column | what it means |
|---|---|
| `name` | the player's name |
| `season` | the year |
| `team` | his team that season |
| `G` | games played |
| `H` | hits |
| `HR` | home runs |
| `RBI` | runs batted in |
| `SB` | stolen bases |
| `BB` | walks |
| `SO` | strikeouts |

### ⚾ Your turn
How many rows are in the box? Python has a command that counts things.
It's 3 letters long and short for "length". Write it in place of `___`.

🔎 **Google it:** `python how to count items in a list`

In [ ]:
print("Rows in the box:", ___(stats))

Every row is one player in one season. That's a lot of seasons!

---
## Part 3. Track down your player 🎯
This box keeps only the rows of one player. Read it like this:
**"from `stats`, keep the rows where the name is the same as `name`."**

One more symbol: `=` puts something into a box, but `==` (two of them) **asks a question**: "are these the same?"

In [ ]:
name = "Aaron Judge"   # change to your favorite player, in quotes, spelled exactly right

seasons = stats[stats["name"] == name]

print(name, "has", len(seasons), "rows: one for each season")
seasons[["season", "team", "G", "H", "HR", "RBI", "SB"]]

If it says **0 rows**, check the spelling: `"Aaron Judge"` works, `"aaron judge"` doesn't.
Capital letters matter to Python.

Today's players are written the way MLB writes them: `"Vladimir Guerrero Jr."`, `"Bobby Witt Jr."`.
All names are **without accent marks**, so type `"Julio Rodriguez"`, not `"Julio Rodríguez"`.

### ⚾ Your turn: his best season ever
Every column has a command that finds its **biggest** number. It's 3 letters, the opposite of "min".
Write it in place of `___`. The brackets `()` after it stay.

If your player is fast rather than strong, change `"HR"` to `"SB"` (stolen bases) or `"H"` (hits) first.

🔎 **Google it:** `pandas biggest value in a column`

In [ ]:
stat = "HR"

print("Best season:", seasons[stat].___())

And here is how many he gets in a **normal** season. `mean` means "average": add up all his seasons
and share them out equally. `round( )` cuts off the decimals. Just run it:

In [ ]:
print("Average season:", round(seasons[stat].mean()))

See `<NA>` and a red `TypeError`? `<NA>` means **empty**: Python found no seasons, so the name in Part 3
is spelled differently. Fix it there, run that box again, then come back.

---
## Part 4. The biggest seasons in history 💥
`sort_values` puts all the rows in order. `ascending=False` means **biggest first**
(without it, the smallest would come first). `.head(5)` shows only the top 5.

In [ ]:
stat = "HR"

biggest = stats.sort_values(stat, ascending=False)
biggest[["name", "season", "team", stat]].head(5)

Who's #1? He also has the most home runs in a whole career. 👑

### ⚾ Your turn
Change **only the first line**: replace `"HR"` with `"H"` (keep the quotes).
Run it and see the most hits anyone ever got in one season.

Then try `"RBI"` and `"SO"` in the same spot.

In [ ]:
stat = "HR"   # change this word

biggest = stats.sort_values(stat, ascending=False)
biggest[["name", "season", "team", stat]].head(5)

---
## Part 5. Who's better? ⚔️
We built you a machine called `compare`. You give it two players and a stat,
and it adds up their **whole careers** and tells you who wins.
Run this box once so Python learns the machine (nothing shows up yet, that's normal):

In [ ]:
def compare(player1, player2, stat):
    total1 = stats[stats["name"] == player1][stat].sum()
    total2 = stats[stats["name"] == player2][stat].sum()
    print(player1, ":", total1)
    print(player2, ":", total2)
    if total1 > total2:
        print("Winner:", player1, "🏆")
    else:
        print("Winner:", player2, "🏆")

Now use it. Inside the brackets: first player, second player, the stat. Each one in quotes, with commas between them.

In [ ]:
compare("Aaron Judge", "Shohei Ohtani", "HR")

### ⚾ Your turn: settle an argument
Pick two players you'd argue about and a stat from the table in Part 2.
Replace the names and the stat below, keeping the quotes and the commas.

In [ ]:
compare("Mookie Betts", "Juan Soto", "H")   # change the names and the stat

---
### 🏆 Case closed, Detective!
Today you learned:
- `print( )` shows things, and **quotes** mean "this is text"
- a **variable** is a box with a name: `player = "..."`
- `=` puts into a box, `==` asks "are these the same?"
- `len( )` counts, `.max()` finds the biggest, `.mean()` finds the average
- and you used a machine someone else built: `compare( )`

Next time: we'll look inside that machine and teach Python to make decisions, like an umpire.